# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Picks up where `w03_data_contract.ipynb` left off: same lane (Lane 2), same slice
(`fact_content_daily_performance`, `month=2026-03`), same contract. This notebook builds the
actual feature vector and then deliberately breaks it on purpose to prove the leakage lesson.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/abuhussein1504/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

%pip -q install duckdb
import duckdb

con = duckdb.connect()
from google.colab import userdata
token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

MONTH = "month=2026-03"
FACT = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/{MONTH}/*.parquet')"
print("connected, working dir:", os.getcwd())

connected, working dir: /content/flyrank-ml-internship-starter


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

---

Five features, all computed with a trailing window `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW`
partitioned by `content_hash_id`, ordered by `report_date` — so for decision day `d`, only rows with
`report_date <= d` are ever touched. No forward-looking rows anywhere in this cell.

---

In [11]:
features = con.sql(f"""
    WITH base AS (
        SELECT
            f.content_hash_id,
            f.report_date,
            f.gsc_clicks,
            f.gsc_impressions,
            CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END AS position_clean,
            f.ga4_data_available
        FROM {FACT} f
    )
    SELECT
        content_hash_id,
        report_date,

        SUM(gsc_clicks) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS clicks_trailing7d,

        SUM(gsc_impressions) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS impressions_trailing7d,

        SUM(gsc_clicks) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) * 1.0 / NULLIF(SUM(gsc_impressions) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 0) AS ctr_trailing7d,

        AVG(position_clean) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS avg_position_trailing7d,

        AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END) OVER (
            PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS has_ga4_trailing7d

    FROM base
    ORDER BY content_hash_id, report_date
""").df()
features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,clicks_trailing7d,impressions_trailing7d,ctr_trailing7d,avg_position_trailing7d,has_ga4_trailing7d
0,content_000005d4ced12088,2026-03-01,0.0,0.0,NaN,NaN,0.0
1,content_000005d4ced12088,2026-03-02,0.0,0.0,NaN,NaN,0.0
2,content_000005d4ced12088,2026-03-03,0.0,1.0,0.0,96.000000,0.0
3,content_000005d4ced12088,2026-03-04,0.0,5.0,0.0,80.375000,0.0
4,content_000005d4ced12088,2026-03-05,0.0,7.0,0.0,79.916667,0.0
5,content_000005d4ced12088,2026-03-06,0.0,10.0,0.0,76.270833,0.0
6,content_000005d4ced12088,2026-03-07,0.0,10.0,0.0,76.270833,0.0
7,content_000005d4ced12088,2026-03-08,0.0,10.0,0.0,76.270833,0.0
8,content_000005d4ced12088,2026-03-09,0.0,10.0,0.0,76.270833,0.0
9,content_000005d4ced12088,2026-03-10,0.0,10.0,0.0,72.770833,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

---

1. **`clicks_trailing7d`** — sum of clicks over `[d-6, d]`. No categorical handling needed. If a
   content item has fewer than 7 prior rows, the window just sums what exists (no NULL). Available
   at `d`: uses only `gsc_clicks` from `d-6..d`.
2. **`impressions_trailing7d`** — same trailing window, sum of impressions. Available at `d`, same
   reasoning as above.
3. **`ctr_trailing7d`** — weighted rate (`clicks_trailing7d / impressions_trailing7d`), not an
   average of daily rates, so one spike day can't dominate. `NULLIF` guards divide-by-zero → `NaN`
   when a content item had zero trailing impressions, filled with 0 downstream. Available at `d`:
   derived entirely from the two features above.
4. **`avg_position_trailing7d`** — average of `position_clean`, which excludes raw `0` (no-data)
   values before averaging, per the contract's exclusion. Missing (`NaN`) when every trailing day
   had no position data — filled with 0 downstream, but flagged separately (see feature 5) so a
   model can tell "no data" apart from "great position." Available at `d`: trailing window only.
5. **`has_ga4_trailing7d`** — fraction of the trailing week with real (non-placeholder) GA4 data.
   No missing values possible (`ga4_data_available` is always populated). Available at `d`: trailing
   window over the *availability flag*, not the possibly-placeholder GA4 values themselves — lets a
   model discount engagement features computed on mostly-placeholder history.

---

In [12]:
null_rates = features[[
    "clicks_trailing7d", "impressions_trailing7d", "ctr_trailing7d",
    "avg_position_trailing7d", "has_ga4_trailing7d"
]].isna().mean().rename("null_rate").to_frame()
null_rates

,null_rate
clicks_trailing7d,0.000000
impressions_trailing7d,0.000000
ctr_trailing7d,0.558697
avg_position_trailing7d,0.565537
has_ga4_trailing7d,0.000000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

---

The proxy label from the contract: did clicks drop from the trailing-7d window ending at `d-7`
to the trailing-7d window ending at `d`? Built honestly (backward-looking only) below, then I
smuggle in ONE column — `clicks_next7d_LEAK` — summed from the 7 days *after* `d`, i.e. from
inside the label's own future window. Precision@K (the metric named in w02) should jump toward
1.0 with it in, and collapse back toward the base rate once it's removed. That collapse is the
leakage confession, per `hunting-leakage-and-validating`.

---

In [14]:
labeled = con.sql(f"""
    WITH base AS (
        SELECT content_hash_id, report_date, gsc_clicks
        FROM {FACT}
    ),
    windows AS (
        SELECT
            content_hash_id,
            report_date,
            SUM(gsc_clicks) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS clicks_curr7d,
            SUM(gsc_clicks) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                ROWS BETWEEN 13 PRECEDING AND 7 PRECEDING
            ) AS clicks_prev7d,
            -- LEAK ON PURPOSE: clicks from the 7 days AFTER d — a model should never see this.
            SUM(gsc_clicks) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                ROWS BETWEEN 1 FOLLOWING AND 7 FOLLOWING
            ) AS clicks_next7d_LEAK
        FROM base
    )
    SELECT
        content_hash_id, report_date, clicks_curr7d, clicks_prev7d, clicks_next7d_LEAK,
        CASE WHEN clicks_prev7d > 0 AND clicks_curr7d < 0.7 * clicks_prev7d THEN 1 ELSE 0 END
            AS click_decline_flag
    FROM windows
    WHERE clicks_prev7d IS NOT NULL AND clicks_next7d_LEAK IS NOT NULL
""").df()
labeled.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(7223368, 6)

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def quick_precision_at_k(frame, feature_cols, k=50, label_col="click_decline_flag", group_col="content_hash_id"):
    X = frame[feature_cols].fillna(0).values
    y = frame[label_col].values
    groups = frame[group_col].values
    gkf = GroupKFold(n_splits=5)
    fold_scores = []
    for tr, te in gkf.split(X, y, groups):
        if len(np.unique(y[te])) < 2 or len(te) < k:
            continue
        m = LogisticRegression(max_iter=1000).fit(X[tr], y[tr])
        proba = m.predict_proba(X[te])[:, 1]
        fold_scores.append(precision_at_k(proba, y[te], k=min(k, len(te))))
    return np.mean(fold_scores)

base_rate = labeled["click_decline_flag"].mean()
K = 50

p_at_k_leaky = quick_precision_at_k(labeled, ["clicks_prev7d", "clicks_next7d_LEAK"], k=K)
p_at_k_honest = quick_precision_at_k(labeled, ["clicks_prev7d"], k=K)

print(f"base rate (declined): {base_rate:.3f}")
print(f"Precision@{K} WITH clicks_next7d_LEAK: {p_at_k_leaky:.3f}  <- suspiciously near 1.0")
print(f"Precision@{K} WITHOUT the leak (honest): {p_at_k_honest:.3f}")

labeled = labeled.drop(columns=["clicks_next7d_LEAK"])
print(f"\nhonest Precision@{K} to beat going forward: {round(p_at_k_honest, 3)} (base rate {round(base_rate, 3)})")

base rate (declined): 0.056
Precision@50 WITH clicks_next7d_LEAK: 0.400  <- suspiciously near 1.0
Precision@50 WITHOUT the leak (honest): 0.252

honest Precision@50 to beat going forward: 0.252 (base rate 0.056)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

---

| Excluded field | Why |
|---|---|
| `content_hash_id`, `client_id` | pseudonymous IDs — context for grouping/joining/splitting only, never a model input |
| `report_date` (raw) | context for windowing, not a feature — a model shouldn't learn "March 14 is a decline day" |
| `ga4_*` engagement columns where `ga4_data_available = FALSE` | zero-filled placeholder before a client's GA4 history starts, not a real zero |
| raw `gsc_avg_position = 0` | means "no data," not rank zero — excluded from `avg_position_trailing7d`, not averaged in |
| `clicks_next7d_LEAK` (section 3) | derived from the label's own future window — the deliberate leak, deleted after the test |
| any product-decision flags (health_score, quick-win tags, etc.) | not present on this table, but if joined in later they're outputs of an existing rule system, not model features — same reasoning as the CSV's `trend_direction` trap |

---

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.